# 🤖 LangGraph Agent Demo
**Project 6 — Relationship Manager Copilot | Phase 4: Agents**

This notebook demonstrates the ReAct agent powered by LangGraph.
It uses the tools we built in Phase 3 to solve complex RM queries.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

Environment loaded.


## 1. Import the Agent Graph
We load the compiled `copilot_app` from our `src.agent` module.

In [2]:
from langchain_core.messages import HumanMessage
from src.agent import copilot_app

def run_agent(query: str, client_id: str, rm_tier: str):
    print(f"\n{'='*80}")
    print(f"USER: {query}")
    print(f"CONTEXT: Client {client_id} | RM Tier {rm_tier}")
    print(f"{'='*80}\n")
    
    # Initialise the state
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "client_id": client_id,
        "rm_tier": rm_tier,
        "current_step": 0,
    }
    
    # Stream the graph execution
    for event in copilot_app.stream(initial_state):
        for node_name, state_update in event.items():
            if node_name == "agent":
                msg = state_update["messages"][-1]
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    print("\n[AGENT] Using tools:")
                    for tc in msg.tool_calls:
                        print(f"  ▶ {tc['name']}({tc['args']})")
                else:
                    print(f"\n[AGENT FINAL RESPONSE]\n{msg.content}\n")
            elif node_name == "tools":
                messages = state_update["messages"]
                print("[TOOLS] Returned data.")

C:\Users\Good\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Test 1: Portfolio & Market Data Combo
The agent should first fetch the client's portfolio, then look up market data for their holdings to provide an answer.

In [3]:
run_agent(
    query="What is my client's current allocation, and how did their VTI holding perform over the last 3 months?",
    client_id="C-204",
    rm_tier="premium"
)


USER: What is my client's current allocation, and how did their VTI holding perform over the last 3 months?
CONTEXT: Client C-204 | RM Tier premium


[AGENT] Using tools:
  ▶ portfolio_lookup_tool({'client_id': 'C-204', 'rm_tier': 'premium'})
  ▶ market_data_tool({'ticker': 'VTI', 'period': '3M'})
[TOOLS] Returned data.

[AGENT FINAL RESPONSE]
Your client, Sarah Chen, has the following current allocation in her portfolio:

- **Equity**: 50%
  - Vanguard Total Stock Market ETF (VTI): 35%
  - Invesco QQQ Trust (QQQ): 15%
- **Fixed Income**: 30%
  - iShares Core U.S. Aggregate Bond ETF (AGG): 20%
  - Vanguard Total Bond Market ETF (BND): 10%
- **Real Estate**: 10%
  - iShares Global REIT ETF (REET): 10%
- **Alternatives**: 5%
  - Vanguard Real Estate Index Fund (VGSLX): 5%
- **Cash**: 5%

Regarding the performance of the Vanguard Total Stock Market ETF (VTI) over the last 3 months, it has returned 5.43%. The current price of VTI is $248.32, and it has an annualized volatility of 14.2%.



## 3. Test 2: Entitlement Block
We ask a standard RM to fetch info for an institutional client. The agent should be blocked by the tool and relay that clearly.

In [4]:
run_agent(
    query="I want to review the portfolio for James Rodriguez.",
    client_id="C-301",
    rm_tier="standard"
)


USER: I want to review the portfolio for James Rodriguez.
CONTEXT: Client C-301 | RM Tier standard


[AGENT] Using tools:
  ▶ portfolio_lookup_tool({'client_id': 'C-301', 'rm_tier': 'standard'})
[TOOLS] Returned data.

[AGENT FINAL RESPONSE]
I'm unable to access the portfolio for James Rodriguez because it requires an 'institutional' tier entitlement, which is beyond the current 'standard' tier access. If you need further assistance, please consult with a colleague who has the appropriate access level.



## 4. Test 3: Compliance & Suitability (The Big Test)
We ask the agent if we can recommend Emerging Markets to Margaret (a conservative client). 
It should:
1. Look up Margaret's profile (conservative).
2. Run the suitability checker.
3. See that the checker returns UNSUITABLE.
4. Refuse the recommendation and quote the policy.

In [5]:
run_agent(
    query="Margaret has some cash to deploy. Can we recommend adding a 10% allocation to the Emerging Markets Equity Fund to boost yields?",
    client_id="C-115",
    rm_tier="premium"
)


USER: Margaret has some cash to deploy. Can we recommend adding a 10% allocation to the Emerging Markets Equity Fund to boost yields?
CONTEXT: Client C-115 | RM Tier premium


[AGENT] Using tools:
  ▶ portfolio_lookup_tool({'client_id': 'C-115', 'rm_tier': 'premium'})
[TOOLS] Returned data.

[AGENT] Using tools:
  ▶ rag_retriever_tool({'query': 'Emerging Markets Equity Fund', 'rm_tier': 'premium', 'doc_types': 'product', 'top_k': 3})
  Embedded 1/1 texts
  Embedding complete: 1 vectors generated.
[TOOLS] Returned data.

[AGENT] Using tools:
  ▶ suitability_checker_tool({'recommendation': 'Add 10% allocation to Emerging Markets Equity Fund', 'client_risk_profile': 'conservative'})
[TOOLS] Returned data.

[AGENT FINAL RESPONSE]
We cannot recommend adding a 10% allocation to the Emerging Markets Equity Fund for Margaret. According to our suitability policy, emerging market investments are prohibited for conservative clients like Margaret. This recommendation is deemed unsuitable due to t